<a href="https://www.kaggle.com/code/lalit7881/bangla-daily-news-corpus-eda-90-33?scriptVersionId=353624939" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
# Suppress warnings and set up the plotting backend
import warnings
warnings.filterwarnings('ignore')
import matplotlib
matplotlib.use('Agg')  # use Agg backend for matplotlib
import matplotlib.pyplot as plt
%matplotlib inline

import seaborn as sns

from pathlib import Path

# Scikit-learn libraries for prediction modeling
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, roc_curve, auc
from sklearn.inspection import permutation_importance

sns.set_style('whitegrid')
plt.switch_backend('Agg')  # ensure plt backend is set correctly when using plt module directly
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/madhavw/travel-and-tourism/Travel And Tourism.csv


## Loading dataset

In [2]:
# Helper function to dynamically resolve the data file path
DATA_ROOTS = [
    Path('/kaggle/input'),
    Path('/kaggle/input'),
    Path('/kaggle/input/datasets')
]

def resolve_data_file(file_name=None, suffixes=None):
    suffixes = tuple(suffixes or [])
    for root in DATA_ROOTS:
        if not root.exists():
            continue
        matches = [p for p in root.rglob('*') if p.is_file()]
        if file_name:
            exact = [p for p in matches if p.name == file_name]
            if exact:
                return exact[0]
        if suffixes:
            typed = [p for p in matches if p.suffix.lower() in suffixes]
            if typed:
                return typed[0]
    raise FileNotFoundError(f'Could not resolve data file: {file_name or suffixes}')

# Resolve the specific csv file for Travel and Tourism data
data_file = resolve_data_file(file_name='Travel And Tourism.csv', suffixes=['.csv'])
print(f'Data file resolved to: {data_file}') 

Data file resolved to: /kaggle/input/datasets/madhavw/travel-and-tourism/Travel And Tourism.csv


In [3]:
df = pd.read_csv(data_file, encoding='UTF-8-SIG', delimiter=',')
print('Data loaded successfully')


Data loaded successfully


In [4]:
df.head()

,Booking_ID,Booking_Date,Travel_Date,Return_Date,Customer_ID,Customer_Name,Customer_Age,Gender,Customer_City,Customer_Country,...,Meal_Plan,Payment_Method,Booking_Status,Cancellation_Status,Cancellation_Reason,Discount_Amount,Customer_Rating,Customer_Review_Comment,Total_Trip_Cost,Transportation_Type
0,1,2026-02-23,2026-04-18,2026-04-21,cust01,Kenzo Vaughn,18,Male,Dubai,United Arab Emirates,...,Breakfast Included,Net Banking,Completed,Not Cancelled,Not Applicable,12800,1.0,Poor service experience,115000,Flight
1,2,2025-05-30,2025-06-29,2025-07-02,cust02,Priya Carter,20,Female,Udaipur,India,...,Full Board,Net Banking,Completed,Not Cancelled,Not Applicable,0,1.5,Poor service experience,56800,Train
2,3,2024-12-30,2025-03-14,2025-03-23,cust03,Elina Vaughn,31,Female,Dubai,United Arab Emirates,...,Full Board,UPI,Completed,Not Cancelled,Not Applicable,74400,2.0,Several arrangements need improvement,669800,Flight
3,4,2026-06-17,2026-09-19,2026-09-26,cust04,Mei Ferraro,30,Male,Kathmandu,Nepal,...,Room Only,Credit Card,Confirmed,Not Cancelled,Not Applicable,4500,0.0,Not yet travelled,84700,Flight
4,5,2025-03-21,2025-04-21,2025-04-24,cust05,Alina Desai,39,Female,Kuala Lumpur,Malaysia,...,Room Only,Digital Wallet,Completed,Not Cancelled,Not Applicable,2800,3.6,Good overall experience,15600,Flight


In [5]:
df.tail()

,Booking_ID,Booking_Date,Travel_Date,Return_Date,Customer_ID,Customer_Name,Customer_Age,Gender,Customer_City,Customer_Country,...,Meal_Plan,Payment_Method,Booking_Status,Cancellation_Status,Cancellation_Reason,Discount_Amount,Customer_Rating,Customer_Review_Comment,Total_Trip_Cost,Transportation_Type
995,996,2024-06-10,2024-06-22,2024-06-29,cust996,Sofia Darwish,23,Female,Indore,India,...,Room Only,UPI,Completed,Not Cancelled,Not Applicable,0,2.9,Average experience,251100,Flight
996,997,2024-11-28,2025-02-06,2025-02-10,cust997,Ronan Noorani,42,Male,Sydney,Australia,...,Half Board,UPI,Completed,Not Cancelled,Not Applicable,4200,3.9,Good overall experience,79000,Flight
997,998,2026-05-18,2026-09-01,2026-09-04,cust998,Lyra Bakshi,50,Female,Colombo,Sri Lanka,...,Half Board,EMI,Completed,Not Cancelled,Not Applicable,0,4.7,Wonderful travel experience,151700,Flight
998,999,2024-08-02,2024-10-22,2024-10-27,cust999,Jonah Mori,21,Female,Nagpur,India,...,Breakfast Included,UPI,Completed,Not Cancelled,Not Applicable,0,3.5,Enjoyable trip with minor delays,238300,Train
999,1000,2025-01-29,2025-05-01,2025-05-06,cust1000,Kai Romero,40,Male,Udaipur,India,...,Breakfast Included,Debit Card,Completed,Not Cancelled,Not Applicable,40200,1.8,The trip did not meet expectations,227700,Flight


In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 29 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Booking_ID               1000 non-null   int64  
 1   Booking_Date             1000 non-null   object 
 2   Travel_Date              1000 non-null   object 
 3   Return_Date              1000 non-null   object 
 4   Customer_ID              1000 non-null   object 
 5   Customer_Name            1000 non-null   object 
 6   Customer_Age             1000 non-null   int64  
 7   Gender                   1000 non-null   object 
 8   Customer_City            1000 non-null   object 
 9   Customer_Country         1000 non-null   object 
 10  Destination_City         1000 non-null   object 
 11  Destination_Country      1000 non-null   object 
 12  Number_of_Travellers     1000 non-null   int64  
 13  Number_of_Adults         1000 non-null   int64  
 14  Number_of_Children       

In [7]:
df.describe()

,Booking_ID,Customer_Age,Number_of_Travellers,Number_of_Adults,Number_of_Children,Hotel_Rating,Number_of_Rooms,Number_of_Nights,Discount_Amount,Customer_Rating,Total_Trip_Cost
count,1000.000000,1000.000000,1000.0000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1000.000000,1.000000e+03
mean,500.500000,34.468000,5.6610,4.295000,1.366000,2.948600,2.959000,5.375000,15980.000000,2.315000,2.278389e+05
std,288.819436,9.327914,3.3189,2.342106,1.593293,1.299325,1.534821,2.262188,24817.542796,1.673941,2.268066e+05
min,1.000000,18.000000,1.0000,1.000000,0.000000,1.000000,1.000000,2.000000,0.000000,0.000000,1.100000e+04
25%,250.750000,26.000000,3.0000,2.000000,0.000000,1.800000,2.000000,3.000000,0.000000,1.000000,8.160000e+04
50%,500.500000,34.000000,5.0000,4.000000,1.000000,2.900000,3.000000,5.000000,7850.000000,2.300000,1.572000e+05
75%,750.250000,42.000000,8.0000,5.000000,3.000000,4.200000,4.000000,7.000000,20125.000000,3.900000,2.832500e+05
max,1000.000000,50.000000,15.0000,15.000000,7.000000,5.000000,8.000000,10.000000,289300.000000,5.000000,1.639200e+06


In [8]:
df.isnull().sum()

Booking_ID                 0
Booking_Date               0
Travel_Date                0
Return_Date                0
Customer_ID                0
Customer_Name              0
Customer_Age               0
Gender                     0
Customer_City              0
Customer_Country           0
Destination_City           0
Destination_Country        0
Number_of_Travellers       0
Number_of_Adults           0
Number_of_Children         0
Hotel_Name                 0
Hotel_Rating               0
Number_of_Rooms            0
Number_of_Nights           0
Meal_Plan                  0
Payment_Method             0
Booking_Status             0
Cancellation_Status        0
Cancellation_Reason        0
Discount_Amount            0
Customer_Rating            0
Customer_Review_Comment    0
Total_Trip_Cost            0
Transportation_Type        0
dtype: int64

In [9]:
df.duplicated().sum()

np.int64(0)

In [10]:
df.dtypes

Booking_ID                   int64
Booking_Date                object
Travel_Date                 object
Return_Date                 object
Customer_ID                 object
Customer_Name               object
Customer_Age                 int64
Gender                      object
Customer_City               object
Customer_Country            object
Destination_City            object
Destination_Country         object
Number_of_Travellers         int64
Number_of_Adults             int64
Number_of_Children           int64
Hotel_Name                  object
Hotel_Rating               float64
Number_of_Rooms              int64
Number_of_Nights             int64
Meal_Plan                   object
Payment_Method              object
Booking_Status              object
Cancellation_Status         object
Cancellation_Reason         object
Discount_Amount              int64
Customer_Rating            float64
Customer_Review_Comment     object
Total_Trip_Cost              int64
Transportation_Type 

In [11]:
df.shape

(1000, 29)

In [12]:
df.columns

Index(['Booking_ID', 'Booking_Date', 'Travel_Date', 'Return_Date',
       'Customer_ID', 'Customer_Name', 'Customer_Age', 'Gender',
       'Customer_City', 'Customer_Country', 'Destination_City',
       'Destination_Country', 'Number_of_Travellers', 'Number_of_Adults',
       'Number_of_Children', 'Hotel_Name', 'Hotel_Rating', 'Number_of_Rooms',
       'Number_of_Nights', 'Meal_Plan', 'Payment_Method', 'Booking_Status',
       'Cancellation_Status', 'Cancellation_Reason', 'Discount_Amount',
       'Customer_Rating', 'Customer_Review_Comment', 'Total_Trip_Cost',
       'Transportation_Type'],
      dtype='object')

## EDA

In [13]:
date_columns = [
    "Booking_Date",
    "Travel_Date",
    "Return_Date"
]

for col in date_columns:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

print("\nDate columns converted successfully.")


# ============================================================
# 5. CREATE USEFUL TIME FEATURES
# ============================================================

if "Booking_Date" in df.columns:

    df["Booking_Year"] = df["Booking_Date"].dt.year
    df["Booking_Month"] = df["Booking_Date"].dt.month
    df["Booking_Month_Name"] = df["Booking_Date"].dt.month_name()
    df["Booking_Day"] = df["Booking_Date"].dt.day
    df["Booking_Day_Name"] = df["Booking_Date"].dt.day_name()

if "Travel_Date" in df.columns:

    df["Travel_Year"] = df["Travel_Date"].dt.year
    df["Travel_Month"] = df["Travel_Date"].dt.month
    df["Travel_Month_Name"] = df["Travel_Date"].dt.month_name()

# Booking lead time
if "Booking_Date" in df.columns and "Travel_Date" in df.columns:
    df["Booking_Lead_Days"] = (
        df["Travel_Date"] - df["Booking_Date"]
    ).dt.days


# ============================================================
# 6. NUMERICAL COLUMN DISTRIBUTIONS
# ============================================================

numeric_columns = [
    "Customer_Age",
    "Number_of_Travellers",
    "Number_of_Adults",
    "Number_of_Children",
    "Hotel_Rating",
    "Number_of_Rooms",
    "Number_of_Nights",
    "Discount_Amount",
    "Customer_Rating",
    "Total_Trip_Cost"
]

numeric_columns = [
    col for col in numeric_columns
    if col in df.columns
]

for col in numeric_columns:

    plt.figure(figsize=(12, 6))

    sns.histplot(
        data=df,
        x=col,
        kde=True,
        bins=30,
        color="royalblue",
        edgecolor="black"
    )

    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()


# ============================================================
# 7. BOXPLOTS FOR OUTLIER ANALYSIS
# ============================================================

for col in numeric_columns:

    plt.figure(figsize=(12, 4))

    sns.boxplot(
        x=df[col],
        color="orange",
        linewidth=2
    )

    plt.title(f"Boxplot - {col}")
    plt.xlabel(col)
    plt.tight_layout()
    plt.show()


# ============================================================
# 8. GENDER DISTRIBUTION
# ============================================================

if "Gender" in df.columns:

    plt.figure(figsize=(8, 6))

    gender_counts = df["Gender"].value_counts()

    plt.pie(
        gender_counts.values,
        labels=gender_counts.index,
        autopct="%1.1f%%",
        startangle=90,
        colors=["#3498db", "#e74c3c", "#2ecc71"],
        wedgeprops={"edgecolor": "black"}
    )

    plt.title("Customer Gender Distribution")
    plt.show()


# ============================================================
# 9. BOOKING STATUS
# ============================================================

if "Booking_Status" in df.columns:

    plt.figure(figsize=(10, 6))

    order = df["Booking_Status"].value_counts().index

    sns.countplot(
        data=df,
        x="Booking_Status",
        order=order,
        hue="Booking_Status",
        palette="Set2",
        legend=False
    )

    plt.title("Booking Status Distribution")
    plt.xlabel("Booking Status")
    plt.ylabel("Number of Bookings")
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()


# ============================================================
# 10. CANCELLATION STATUS
# ============================================================

if "Cancellation_Status" in df.columns:

    plt.figure(figsize=(10, 6))

    order = df["Cancellation_Status"].value_counts().index

    sns.countplot(
        data=df,
        x="Cancellation_Status",
        order=order,
        hue="Cancellation_Status",
        palette="coolwarm",
        legend=False
    )

    plt.title("Cancellation Status Distribution")
    plt.xlabel("Cancellation Status")
    plt.ylabel("Number of Bookings")
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()


# ============================================================
# 11. PAYMENT METHOD DISTRIBUTION
# ============================================================

if "Payment_Method" in df.columns:

    plt.figure(figsize=(10, 6))

    payment_counts = df["Payment_Method"].value_counts()

    sns.barplot(
        x=payment_counts.index,
        y=payment_counts.values,
        hue=payment_counts.index,
        palette="viridis",
        legend=False
    )

    plt.title("Bookings by Payment Method")
    plt.xlabel("Payment Method")
    plt.ylabel("Number of Bookings")
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()


# ============================================================
# 12. TRANSPORTATION TYPE
# ============================================================

if "Transportation_Type" in df.columns:

    plt.figure(figsize=(10, 6))

    transport_counts = df["Transportation_Type"].value_counts()

    sns.barplot(
        x=transport_counts.index,
        y=transport_counts.values,
        hue=transport_counts.index,
        palette="Set1",
        legend=False
    )

    plt.title("Bookings by Transportation Type")
    plt.xlabel("Transportation Type")
    plt.ylabel("Number of Bookings")
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()


# ============================================================
# 13. MEAL PLAN DISTRIBUTION
# ============================================================

if "Meal_Plan" in df.columns:

    plt.figure(figsize=(10, 6))

    meal_counts = df["Meal_Plan"].value_counts()

    sns.barplot(
        x=meal_counts.index,
        y=meal_counts.values,
        hue=meal_counts.index,
        palette="magma",
        legend=False
    )

    plt.title("Bookings by Meal Plan")
    plt.xlabel("Meal Plan")
    plt.ylabel("Number of Bookings")
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()


# ============================================================
# 14. TOP CUSTOMER CITIES
# ============================================================

if "Customer_City" in df.columns:

    top_cities = df["Customer_City"].value_counts().head(15)

    plt.figure(figsize=(12, 7))

    sns.barplot(
        x=top_cities.values,
        y=top_cities.index,
        hue=top_cities.index,
        palette="Blues_r",
        legend=False
    )

    plt.title("Top 15 Customer Cities")
    plt.xlabel("Number of Bookings")
    plt.ylabel("Customer City")
    plt.tight_layout()
    plt.show()


# ============================================================
# 15. TOP DESTINATION CITIES
# ============================================================

if "Destination_City" in df.columns:

    top_destinations = df["Destination_City"].value_counts().head(15)

    plt.figure(figsize=(12, 7))

    sns.barplot(
        x=top_destinations.values,
        y=top_destinations.index,
        hue=top_destinations.index,
        palette="Greens_r",
        legend=False
    )

    plt.title("Top 15 Destination Cities")
    plt.xlabel("Number of Bookings")
    plt.ylabel("Destination City")
    plt.tight_layout()
    plt.show()


# ============================================================
# 16. TOP DESTINATION COUNTRIES
# ============================================================

if "Destination_Country" in df.columns:

    top_countries = df["Destination_Country"].value_counts().head(15)

    plt.figure(figsize=(12, 7))

    sns.barplot(
        x=top_countries.values,
        y=top_countries.index,
        hue=top_countries.index,
        palette="YlOrRd",
        legend=False
    )

    plt.title("Top 15 Destination Countries")
    plt.xlabel("Number of Bookings")
    plt.ylabel("Destination Country")
    plt.tight_layout()
    plt.show()


# ============================================================
# 17. HOTEL RATING DISTRIBUTION
# ============================================================

if "Hotel_Rating" in df.columns:

    plt.figure(figsize=(10, 6))

    sns.countplot(
        data=df,
        x="Hotel_Rating",
        hue="Hotel_Rating",
        palette="Spectral",
        legend=False
    )

    plt.title("Hotel Rating Distribution")
    plt.xlabel("Hotel Rating")
    plt.ylabel("Number of Bookings")
    plt.tight_layout()
    plt.show()


# ============================================================
# 18. CUSTOMER RATING DISTRIBUTION
# ============================================================

if "Customer_Rating" in df.columns:

    plt.figure(figsize=(10, 6))

    sns.countplot(
        data=df,
        x="Customer_Rating",
        hue="Customer_Rating",
        palette="cool",
        legend=False
    )

    plt.title("Customer Rating Distribution")
    plt.xlabel("Customer Rating")
    plt.ylabel("Number of Customers")
    plt.tight_layout()
    plt.show()


# ============================================================
# 19. AGE GROUP ANALYSIS
# ============================================================

if "Customer_Age" in df.columns:

    df["Age_Group"] = pd.cut(
        df["Customer_Age"],
        bins=[0, 18, 25, 35, 45, 55, 65, 100],
        labels=[
            "Under 18",
            "18-25",
            "26-35",
            "36-45",
            "46-55",
            "56-65",
            "65+"
        ]
    )

    plt.figure(figsize=(12, 6))

    sns.countplot(
        data=df,
        x="Age_Group",
        hue="Age_Group",
        palette="plasma",
        legend=False
    )

    plt.title("Customer Age Group Distribution")
    plt.xlabel("Age Group")
    plt.ylabel("Number of Customers")
    plt.xticks(rotation=20)
    plt.tight_layout()
    plt.show()


# ============================================================
# 20. NUMBER OF TRAVELLERS
# ============================================================

if "Number_of_Travellers" in df.columns:

    plt.figure(figsize=(12, 6))

    traveller_counts = df["Number_of_Travellers"].value_counts().sort_index()

    sns.barplot(
        x=traveller_counts.index,
        y=traveller_counts.values,
        hue=traveller_counts.index,
        palette="cubehelix",
        legend=False
    )

    plt.title("Number of Travellers per Booking")
    plt.xlabel("Number of Travellers")
    plt.ylabel("Bookings")
    plt.tight_layout()
    plt.show()


# ============================================================
# 21. NUMBER OF NIGHTS
# ============================================================

if "Number_of_Nights" in df.columns:

    plt.figure(figsize=(12, 6))

    sns.histplot(
        data=df,
        x="Number_of_Nights",
        bins=20,
        kde=True,
        color="purple",
        edgecolor="black"
    )

    plt.title("Distribution of Number of Nights")
    plt.xlabel("Number of Nights")
    plt.ylabel("Number of Bookings")
    plt.tight_layout()
    plt.show()


# ============================================================
# 22. TOTAL TRIP COST DISTRIBUTION
# ============================================================

if "Total_Trip_Cost" in df.columns:

    plt.figure(figsize=(12, 6))

    sns.histplot(
        data=df,
        x="Total_Trip_Cost",
        bins=30,
        kde=True,
        color="green",
        edgecolor="black"
    )

    plt.title("Distribution of Total Trip Cost")
    plt.xlabel("Total Trip Cost")
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()


# ============================================================
# 23. DISCOUNT AMOUNT DISTRIBUTION
# ============================================================

if "Discount_Amount" in df.columns:

    plt.figure(figsize=(12, 6))

    sns.histplot(
        data=df,
        x="Discount_Amount",
        bins=30,
        kde=True,
        color="red",
        edgecolor="black"
    )

    plt.title("Distribution of Discount Amount")
    plt.xlabel("Discount Amount")
    plt.ylabel("Frequency")
    plt.tight_layout()
    plt.show()


# ============================================================
# 24. AGE VS TOTAL TRIP COST
# ============================================================

if "Customer_Age" in df.columns and "Total_Trip_Cost" in df.columns:

    plt.figure(figsize=(12, 7))

    sns.scatterplot(
        data=df,
        x="Customer_Age",
        y="Total_Trip_Cost",
        hue="Gender" if "Gender" in df.columns else None,
        palette="Set1",
        alpha=0.7,
        s=70
    )

    plt.title("Customer Age vs Total Trip Cost")
    plt.xlabel("Customer Age")
    plt.ylabel("Total Trip Cost")
    plt.tight_layout()
    plt.show()


# ============================================================
# 25. NUMBER OF NIGHTS VS TOTAL COST
# ============================================================

if "Number_of_Nights" in df.columns and "Total_Trip_Cost" in df.columns:

    plt.figure(figsize=(12, 7))

    sns.scatterplot(
        data=df,
        x="Number_of_Nights",
        y="Total_Trip_Cost",
        hue="Booking_Status" if "Booking_Status" in df.columns else None,
        palette="Dark2",
        alpha=0.7,
        s=70
    )

    plt.title("Number of Nights vs Total Trip Cost")
    plt.xlabel("Number of Nights")
    plt.ylabel("Total Trip Cost")
    plt.tight_layout()
    plt.show()


# ============================================================
# 26. HOTEL RATING VS CUSTOMER RATING
# ============================================================

if "Hotel_Rating" in df.columns and "Customer_Rating" in df.columns:

    plt.figure(figsize=(12, 7))

    sns.boxplot(
        data=df,
        x="Hotel_Rating",
        y="Customer_Rating",
        hue="Hotel_Rating",
        palette="Set3",
        legend=False
    )

    plt.title("Hotel Rating vs Customer Rating")
    plt.xlabel("Hotel Rating")
    plt.ylabel("Customer Rating")
    plt.tight_layout()
    plt.show()


# ============================================================
# 27. BOOKING STATUS VS TOTAL TRIP COST
# ============================================================

if "Booking_Status" in df.columns and "Total_Trip_Cost" in df.columns:

    plt.figure(figsize=(12, 7))

    sns.boxplot(
        data=df,
        x="Booking_Status",
        y="Total_Trip_Cost",
        hue="Booking_Status",
        palette="Set2",
        legend=False
    )

    plt.title("Booking Status vs Total Trip Cost")
    plt.xlabel("Booking Status")
    plt.ylabel("Total Trip Cost")
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()


# ============================================================
# 28. CANCELLATION STATUS VS TRIP COST
# ============================================================

if "Cancellation_Status" in df.columns and "Total_Trip_Cost" in df.columns:

    plt.figure(figsize=(12, 7))

    sns.boxplot(
        data=df,
        x="Cancellation_Status",
        y="Total_Trip_Cost",
        hue="Cancellation_Status",
        palette="coolwarm",
        legend=False
    )

    plt.title("Cancellation Status vs Total Trip Cost")
    plt.xlabel("Cancellation Status")
    plt.ylabel("Total Trip Cost")
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()


# ============================================================
# 29. GENDER VS BOOKING STATUS
# ============================================================

if "Gender" in df.columns and "Booking_Status" in df.columns:

    plt.figure(figsize=(12, 7))

    sns.countplot(
        data=df,
        x="Gender",
        hue="Booking_Status",
        palette="Set1"
    )

    plt.title("Gender vs Booking Status")
    plt.xlabel("Gender")
    plt.ylabel("Number of Bookings")
    plt.tight_layout()
    plt.show()


# ============================================================
# 30. PAYMENT METHOD VS BOOKING STATUS
# ============================================================

if "Payment_Method" in df.columns and "Booking_Status" in df.columns:

    plt.figure(figsize=(12, 7))

    sns.countplot(
        data=df,
        x="Payment_Method",
        hue="Booking_Status",
        palette="Paired"
    )

    plt.title("Payment Method vs Booking Status")
    plt.xlabel("Payment Method")
    plt.ylabel("Number of Bookings")
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()


# ============================================================
# 31. TRANSPORTATION VS BOOKING STATUS
# ============================================================

if "Transportation_Type" in df.columns and "Booking_Status" in df.columns:

    plt.figure(figsize=(12, 7))

    sns.countplot(
        data=df,
        x="Transportation_Type",
        hue="Booking_Status",
        palette="Accent"
    )

    plt.title("Transportation Type vs Booking Status")
    plt.xlabel("Transportation Type")
    plt.ylabel("Number of Bookings")
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()


# ============================================================
# 32. MEAL PLAN VS TOTAL TRIP COST
# ============================================================

if "Meal_Plan" in df.columns and "Total_Trip_Cost" in df.columns:

    plt.figure(figsize=(12, 7))

    sns.boxplot(
        data=df,
        x="Meal_Plan",
        y="Total_Trip_Cost",
        hue="Meal_Plan",
        palette="Pastel1",
        legend=False
    )

    plt.title("Meal Plan vs Total Trip Cost")
    plt.xlabel("Meal Plan")
    plt.ylabel("Total Trip Cost")
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()


# ============================================================
# 33. MONTHLY BOOKING TREND
# ============================================================

if "Booking_Date" in df.columns:

    monthly_bookings = (
        df.dropna(subset=["Booking_Date"])
          .set_index("Booking_Date")
          .resample("M")
          .size()
    )

    plt.figure(figsize=(14, 6))

    plt.plot(
        monthly_bookings.index,
        monthly_bookings.values,
        marker="o",
        linewidth=2.5,
        color="blue"
    )

    plt.fill_between(
        monthly_bookings.index,
        monthly_bookings.values,
        alpha=0.2,
        color="blue"
    )

    plt.title("Monthly Booking Trend")
    plt.xlabel("Month")
    plt.ylabel("Number of Bookings")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


# ============================================================
# 34. MONTHLY TRIP COST TREND
# ============================================================

if "Booking_Date" in df.columns and "Total_Trip_Cost" in df.columns:

    monthly_cost = (
        df.dropna(subset=["Booking_Date"])
          .set_index("Booking_Date")["Total_Trip_Cost"]
          .resample("M")
          .sum()
    )

    plt.figure(figsize=(14, 6))

    plt.plot(
        monthly_cost.index,
        monthly_cost.values,
        marker="o",
        linewidth=2.5,
        color="darkgreen"
    )

    plt.fill_between(
        monthly_cost.index,
        monthly_cost.values,
        alpha=0.2,
        color="green"
    )

    plt.title("Monthly Total Trip Cost")
    plt.xlabel("Month")
    plt.ylabel("Total Trip Cost")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()


# ============================================================
# 35. BOOKING LEAD TIME
# ============================================================

if "Booking_Lead_Days" in df.columns:

    plt.figure(figsize=(12, 6))

    sns.histplot(
        data=df,
        x="Booking_Lead_Days",
        bins=30,
        kde=True,
        color="teal",
        edgecolor="black"
    )

    plt.title("Booking Lead Time Distribution")
    plt.xlabel("Days Between Booking and Travel")
    plt.ylabel("Number of Bookings")
    plt.tight_layout()
    plt.show()


# ============================================================
# 36. CORRELATION HEATMAP
# ============================================================

correlation_columns = [
    col for col in numeric_columns
    if col in df.columns
]

# Add lead time if available
if "Booking_Lead_Days" in df.columns:
    correlation_columns.append("Booking_Lead_Days")

corr = df[correlation_columns].corr()

plt.figure(figsize=(14, 10))

sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="RdYlBu_r",
    linewidths=0.5,
    square=True
)

plt.title("Correlation Heatmap of Numerical Variables")
plt.tight_layout()
plt.show()


# ============================================================
# 37. TOP HOTELS
# ============================================================

if "Hotel_Name" in df.columns:

    top_hotels = df["Hotel_Name"].value_counts().head(15)

    plt.figure(figsize=(12, 7))

    sns.barplot(
        x=top_hotels.values,
        y=top_hotels.index,
        hue=top_hotels.index,
        palette="rocket",
        legend=False
    )

    plt.title("Top 15 Most Booked Hotels")
    plt.xlabel("Number of Bookings")
    plt.ylabel("Hotel Name")
    plt.tight_layout()
    plt.show()


# ============================================================
# 38. CANCELLATION REASONS
# ============================================================

if "Cancellation_Reason" in df.columns:

    cancellation_reasons = (
        df["Cancellation_Reason"]
        .dropna()
        .value_counts()
        .head(15)
    )

    if len(cancellation_reasons) > 0:

        plt.figure(figsize=(12, 7))

        sns.barplot(
            x=cancellation_reasons.values,
            y=cancellation_reasons.index,
            hue=cancellation_reasons.index,
            palette="flare",
            legend=False
        )

        plt.title("Top Cancellation Reasons")
        plt.xlabel("Number of Cancellations")
        plt.ylabel("Cancellation Reason")
        plt.tight_layout()
        plt.show()


# ============================================================
# 39. DISCOUNT VS TOTAL COST
# ============================================================

if "Discount_Amount" in df.columns and "Total_Trip_Cost" in df.columns:

    plt.figure(figsize=(12, 7))

    sns.scatterplot(
        data=df,
        x="Discount_Amount",
        y="Total_Trip_Cost",
        hue="Booking_Status" if "Booking_Status" in df.columns else None,
        palette="tab10",
        alpha=0.7,
        s=70
    )

    plt.title("Discount Amount vs Total Trip Cost")
    plt.xlabel("Discount Amount")
    plt.ylabel("Total Trip Cost")
    plt.tight_layout()
    plt.show()


# ============================================================
# 40. ROOMS VS NIGHTS
# ============================================================

if "Number_of_Rooms" in df.columns and "Number_of_Nights" in df.columns:

    plt.figure(figsize=(12, 7))

    sns.scatterplot(
        data=df,
        x="Number_of_Rooms",
        y="Number_of_Nights",
        hue="Booking_Status" if "Booking_Status" in df.columns else None,
        palette="husl",
        alpha=0.7,
        s=70
    )

    plt.title("Number of Rooms vs Number of Nights")
    plt.xlabel("Number of Rooms")
    plt.ylabel("Number of Nights")
    plt.tight_layout()
    plt.show()


# ============================================================
# 41. CUSTOMER RATING VS TOTAL COST
# ============================================================

if "Customer_Rating" in df.columns and "Total_Trip_Cost" in df.columns:

    plt.figure(figsize=(12, 7))

    sns.scatterplot(
        data=df,
        x="Customer_Rating",
        y="Total_Trip_Cost",
        hue="Gender" if "Gender" in df.columns else None,
        palette="Dark2",
        alpha=0.7,
        s=80
    )

    plt.title("Customer Rating vs Total Trip Cost")
    plt.xlabel("Customer Rating")
    plt.ylabel("Total Trip Cost")
    plt.tight_layout()
    plt.show()


# ============================================================
# 42. SUMMARY DASHBOARD
# ============================================================

print("\n" + "=" * 70)
print("                 TRAVEL BOOKING DATASET SUMMARY")
print("=" * 70)

print(f"Total Records       : {len(df):,}")
print(f"Total Columns       : {df.shape[1]:,}")

if "Customer_ID" in df.columns:
    print(f"Unique Customers    : {df['Customer_ID'].nunique():,}")

if "Booking_ID" in df.columns:
    print(f"Unique Bookings     : {df['Booking_ID'].nunique():,}")

if "Destination_City" in df.columns:
    print(
        f"Unique Destinations : "
        f"{df['Destination_City'].nunique():,}"
    )

if "Hotel_Name" in df.columns:
    print(
        f"Unique Hotels       : "
        f"{df['Hotel_Name'].nunique():,}"
    )

if "Total_Trip_Cost" in df.columns:
    print(
        f"Total Trip Revenue  : "
        f"{df['Total_Trip_Cost'].sum():,.2f}"
    )

if "Discount_Amount" in df.columns:
    print(
        f"Total Discounts     : "
        f"{df['Discount_Amount'].sum():,.2f}"
    )

print("=" * 70)
print("\nVisualization analysis completed successfully!")


Date columns converted successfully.

                 TRAVEL BOOKING DATASET SUMMARY
Total Records       : 1,000
Total Columns       : 39
Unique Customers    : 1,000
Unique Bookings     : 1,000
Unique Destinations : 26
Unique Hotels       : 78
Total Trip Revenue  : 227,838,900.00
Total Discounts     : 15,980,000.00

Visualization analysis completed successfully!


In [14]:
numeric_df = df.select_dtypes(include=[np.number])

if numeric_df.shape[1] >= 4:
    plt.figure(figsize=(10,8))
    corr = numeric_df.corr()
    sns.heatmap(corr, annot=True, cmap='coolwarm')
    plt.title('Correlation Heatmap of Numeric Features')
    plt.show()
else:
    print('Not enough numeric columns for correlation heatmap.')

# Pair Plot to observe pairwise relationships amongst a subset of numeric features
if numeric_df.shape[1] >= 2:
    sns.pairplot(numeric_df.iloc[:, :min(5, numeric_df.shape[1])])  # limit to first 5 numeric columns for clarity
    plt.show()

# Histogram for Customer_Age distribution
if 'Customer_Age' in df.columns:
    plt.figure(figsize=(8,6))
    sns.histplot(df['Customer_Age'], kde=True, bins=20)
    plt.title('Distribution of Customer Age')
    plt.xlabel('Customer Age')
    plt.ylabel('Frequency')
    plt.show()

# Count Plot for Gender distribution
if 'Gender' in df.columns:
    plt.figure(figsize=(6,4))
    sns.countplot(x='Gender', data=df)
    plt.title('Count of Customers by Gender')
    plt.xlabel('Gender')
    plt.ylabel('Count')
    plt.show()

# Box Plot for Total_Trip_Cost to identify outliers
if 'Total_Trip_Cost' in df.columns:
    plt.figure(figsize=(8,6))
    sns.boxplot(y='Total_Trip_Cost', data=df)
    plt.title('Box Plot of Total Trip Cost')
    plt.ylabel('Total Trip Cost')
    plt.show()

## Feature engineering

In [15]:
target_col = 'Booking_Status'
if target_col in df.columns:
    # Create a binary target. This mapping can be adjusted according to actual dataset details.
    df['Booking_Status_Binary'] = df[target_col].apply(lambda x: 1 if str(x).strip().lower() == 'confirmed' else 0)

    # Select features for the predictor. We choose numeric features that may have predictive power.
    features = ['Customer_Age', 'Number_of_Travellers', 'Number_of_Adults', 'Number_of_Children',
                'Hotel_Rating', 'Number_of_Rooms', 'Number_of_Nights', 'Discount_Amount', 'Customer_Rating']

    # Remove features that may be missing or not present
    features = [feat for feat in features if feat in df.columns]

    # Drop rows with missing values in these columns
    data = df[features + ['Booking_Status_Binary']].dropna()

    X = data[features]
    y = data['Booking_Status_Binary']

    # Split data into training and testing sets
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

    # Initialize the classifier (Random Forest for example purposes)
    clf = RandomForestClassifier(n_estimators=100, random_state=42)
    clf.fit(X_train, y_train)

    # Perform predictions
    y_pred = clf.predict(X_test)

    # Calculate prediction accuracy
    accuracy = accuracy_score(y_test, y_pred)
    print(f'Prediction Accuracy: {accuracy:.4f}')

    # Plot the Confusion Matrix using a heatmap
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(6,5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title('Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.show()

    # Plot the ROC Curve
    y_prob = clf.predict_proba(X_test)[:, 1]
    fpr, tpr, thresholds = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    plt.figure(figsize=(8,6))
    plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC curve (area = {roc_auc:.2f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('Receiver Operating Characteristic (ROC)')
    plt.legend(loc="lower right")
    plt.show()

    # Permutation Importance to evaluate feature importance
    perm_importance = permutation_importance(clf, X_test, y_test, n_repeats=10, random_state=42)
    sorted_idx = perm_importance.importances_mean.argsort()
    plt.figure(figsize=(8,6))
    plt.barh(range(len(sorted_idx)), perm_importance.importances_mean[sorted_idx])
    plt.yticks(range(len(sorted_idx)), [features[i] for i in sorted_idx])
    plt.xlabel('Mean Permutation Importance')
    plt.title('Permutation Importance of Features')
    plt.show()

else:
    print(f'Target column "{target_col}" not found in the dataset.')

Prediction Accuracy: 0.9033


In [16]:
print("thank you..pls upvote!!!!!!!!!!")

thank you..pls upvote!!!!!!!!!!
